# 参考题解：多头潜在注意力（MLA）

实现一个 CPU 可练习的 MLA 简化版本，通过共享低秩潜变量压缩 K/V。

核心思路：先把输入压缩到低维潜变量，再分别解压得到 K 和 V；Q 保持标准多头投影。


In [ ]:
# ✅ SOLUTION

import torch
import torch.nn as nn
import math

class MultiHeadLatentAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, latent_dim: int):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")
        self.d_model, self.num_heads = d_model, num_heads
        self.head_dim = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.kv_down = nn.Linear(d_model, latent_dim, bias=False)
        self.k_up = nn.Linear(latent_dim, d_model, bias=False)
        self.v_up = nn.Linear(latent_dim, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor, mask=None):
        batch, seq_len, _ = x.shape
        q = self.W_q(x).view(batch,seq_len,self.num_heads,self.head_dim).transpose(1,2)
        latent = self.kv_down(x)
        k = self.k_up(latent).view(batch,seq_len,self.num_heads,self.head_dim).transpose(1,2)
        v = self.v_up(latent).view(batch,seq_len,self.num_heads,self.head_dim).transpose(1,2)
        scores = q @ k.transpose(-2,-1) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        out = torch.softmax(scores,-1) @ v
        return self.W_o(out.transpose(1,2).contiguous().view(batch,seq_len,self.d_model))
